# LeetCode 467 — Unique Substrings in Wraparound String bằng Python

Link đề: https://leetcode.com/problems/unique-substrings-in-wraparound-string/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Đếm theo (chữ cái kết thúc, độ dài)

Skeleton LeetCode:

```python
class Solution:
    def findSubstringInWraproundString(self, s: str) -> int:
        ...
```

Ghi chú: tên method trên LeetCode viết là `Wrapround` (thiếu một chữ `a`), giữ nguyên như vậy. Mọi đáp án trong notebook đều được đối chiếu với một lời giải vét cạn độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `base` là bảng chữ cái `abc...zabc...` lặp **vô hạn**; ta phải đếm số **chuỗi con khác nhau** (không đếm trùng) của `s` mà cũng xuất hiện trong `base`.
2. Nắm kỹ thuật **modulo 26** để kiểm tra "hai chữ cái liên tiếp có kề nhau trong vòng tròn bảng chữ cái không" (kể cả `z → a`), cùng `ord`, `chr`, `enumerate`, `set`.
3. Phát hiện **bước đột phá**: một chuỗi con hợp lệ được xác định hoàn toàn bởi cặp **(chữ cái cuối, độ dài)**; nhờ đó chỉ cần nhớ "độ dài dài nhất kết thúc ở mỗi chữ cái" (mảng 26 phần tử).
4. Đi từ **vét cạn với `set` các chuỗi con (`O(n³)`)** → **mở rộng có cắt nhánh** → **DP theo chữ cái cuối `O(n)`** → **DP theo chữ cái đầu (đối xứng)** → **chia theo đoạn liên tiếp tối đa**, và so sánh chúng.
5. Tránh các lỗi kinh điển: quên `z → a`, cộng dồn `k` thay vì lấy `max` theo chữ cái, quên reset `k = 1`, dùng `set` chuỗi con cho `n = 10^5` (tràn thời gian và bộ nhớ).

## 2. Đề bài & ví dụ

Định nghĩa chuỗi `base` là chuỗi vô hạn lặp lại bảng chữ cái tiếng Anh in thường:

`...zabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcd...`

Cho chuỗi `s`. Hãy trả về **số chuỗi con khác nhau, không rỗng** của `s` mà **cũng là chuỗi con của `base`**.

**Ví dụ 1:** `s = "a"` → `1` (chỉ có `"a"`).

**Ví dụ 2:** `s = "cac"` → `2` (hai chuỗi con `"a"` và `"c"`; còn `"ca"`, `"ac"`, `"cac"` không xuất hiện trong `base`).

**Ví dụ 3:** `s = "zab"` → `6` (sáu chuỗi con `"z"`, `"a"`, `"b"`, `"za"`, `"ab"`, `"zab"`).

**Ràng buộc:** `1 <= s.length <= 10^5`; `s` chỉ gồm chữ cái tiếng Anh in thường.

In [1]:
import random
import string
import time
from collections import defaultdict
from typing import List

examples = [("a", 1), ("cac", 2), ("zab", 6)]
for s, expected in examples:
    print(f"s = {s!r:6} -> đáp án mong đợi {expected}")

s = 'a'    -> đáp án mong đợi 1
s = 'cac'  -> đáp án mong đợi 2
s = 'zab'  -> đáp án mong đợi 6


## 3. Kiến thức nền

### 3.1. `ord`, `chr` và vị trí của chữ cái

Máy tính lưu mỗi ký tự bằng một mã số. `ord('a')` cho mã của `'a'` (97), `ord('b')` là 98... Các chữ cái liên tiếp có mã liên tiếp. Vì vậy `ord(c) - ord('a')` cho **vị trí** của chữ cái trong bảng chữ cái: `a → 0`, `b → 1`, ..., `z → 25`. Hàm `chr` làm ngược lại.

In [2]:
print(ord("a"), ord("b"), ord("z"))              # 97 98 122
print(ord("c") - ord("a"))                       # 2  -> 'c' đứng ở vị trí 2
print(chr(ord("a") + 25))                        # z
print([ord(c) - ord("a") for c in "zab"])        # [25, 0, 1]

97 98 122
2
z
[25, 0, 1]


### 3.2. Phép chia lấy dư `%` và "vòng tròn" 26 chữ cái

Bảng chữ cái là một **vòng tròn**: sau `z` lại đến `a`. Hai chữ cái `p` rồi `c` là "kề nhau đúng thứ tự" nếu `c` đứng ngay sau `p` trong vòng tròn, tức `(vị_trí(c) - vị_trí(p)) % 26 == 1`.

Ví dụ: `a → b`: `(1 - 0) % 26 = 1` ✔. `z → a`: `(0 - 25) % 26 = (-25) % 26 = 1` ✔ (Python cho kết quả **không âm** khi chia dư cho số dương, nên dùng được ngay với số âm). `a → c`: `2` ✘. `b → a`: `(0 - 1) % 26 = 25` ✘.

In [3]:
def ke_nhau(p, c):
    return (ord(c) - ord(p)) % 26 == 1

for p, c in [("a", "b"), ("z", "a"), ("a", "c"), ("b", "a"), ("c", "c")]:
    print(f"{p}{c}: hiệu = {(ord(c) - ord(p)) % 26:2d} -> kề nhau? {ke_nhau(p, c)}")
print((-25) % 26, (-1) % 26)        # 1 và 25 : Python luôn trả về số không âm

ab: hiệu =  1 -> kề nhau? True
za: hiệu =  1 -> kề nhau? True
ac: hiệu =  2 -> kề nhau? False
ba: hiệu = 25 -> kề nhau? False
cc: hiệu =  0 -> kề nhau? False
1 25


### 3.3. Chuỗi con (substring), cắt lát `s[i:j]` và `set`

Chuỗi con là một đoạn **liên tiếp** của `s`. `s[i:j]` lấy các ký tự từ chỉ số `i` đến `j-1` (không gồm `j`). Một `set` chỉ giữ mỗi phần tử một lần nên rất hợp để đếm "số chuỗi con **khác nhau**".

In [4]:
s = "abab"
tat_ca = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print(tat_ca)                    # 10 chuỗi con (tính cả trùng)
print(len(tat_ca), len(set(tat_ca)))     # 10 và 7 (khác nhau): a, b, ab, ba, aba, bab, abab
print(sorted(set(tat_ca)))

['a', 'ab', 'aba', 'abab', 'b', 'ba', 'bab', 'a', 'ab', 'b']
10 7
['a', 'ab', 'aba', 'abab', 'b', 'ba', 'bab']


### 3.4. Mảng đếm 26 ô và `max` để "giữ kỷ lục"

Khi cần nhớ "giá trị lớn nhất đã thấy" cho mỗi chữ cái, ta dùng một list gồm 26 số 0: `f = [0] * 26`. `f[ord(c) - ord('a')]` là ô của chữ cái `c`. Để giữ kỷ lục: `f[idx] = max(f[idx], gia_tri_moi)`.

In [5]:
f = [0] * 26
for c, k in [("a", 3), ("b", 1), ("a", 2), ("a", 5), ("b", 4)]:
    idx = ord(c) - ord("a")
    f[idx] = max(f[idx], k)
print(f[:3], "tổng =", sum(f))        # [5, 4, 0] tổng = 9

[5, 4, 0] tổng = 9


### 3.5. `enumerate` và `sum`

`for i, c in enumerate(s)` cho **cả chỉ số và ký tự** cùng lúc, thay cho `range(len(s))` rồi `s[i]`. `sum(list)` cộng mọi phần tử.

In [6]:
for i, c in enumerate("zab"):
    print(i, c)
print(sum([5, 4, 0]), sum(range(1, 5)))      # 9 10

0 z
1 a
2 b
9 10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Bước 1 — chuỗi con nào nằm trong `base`?** Một chuỗi con của `base` là một đoạn liên tiếp của vòng tròn bảng chữ cái, nên **mỗi ký tự phải đứng ngay sau ký tự trước nó** trong vòng tròn. Tức mọi cặp kề nhau `t[i-1], t[i]` đều phải có `(ord(t[i]) - ord(t[i-1])) % 26 == 1`.

**Bước 2 — tính tay `s = "zab"`.** Các cặp kề: `z→a` ✔, `a→b` ✔. Cả `"zab"` hợp lệ nên mọi chuỗi con của nó đều hợp lệ: `z, a, b, za, ab, zab` = **6**.

**Bước 3 — `s = "cac"`.** `c→a` ✘ (hiệu 24), `a→c` ✘ (hiệu 2). Chỉ các chuỗi một ký tự hợp lệ: `c`, `a`, `c` nhưng chữ `c` lặp → khác nhau chỉ có `{a, c}` = **2**.

**Bước 4 — nhận xét về trùng lặp.** Hãy xét `s = "abcabc"`. Các đoạn liên tiếp hợp lệ tối đa: toàn bộ `"abc"` (vị trí 0–2) rồi `c→a`? Hiệu `(0-2)%26 = 24` ✘; nên `"abc"` và `"abc"` là hai đoạn liên tiếp tối đa **giống hệt nhau**. Ta không được đếm chúng hai lần: đáp án là `a, b, c, ab, bc, abc` = **6**, không phải 12.

**Nhận xét quan trọng:** hai chuỗi con hợp lệ kết thúc **cùng một chữ cái** thì cái **ngắn hơn là hậu tố (đuôi) của cái dài hơn**. Ví dụ `"c"` là đuôi của `"bc"`, và `"bc"` là đuôi của `"abc"`. Vậy nếu ta đã đếm `"abc"` (dài 3, kết thúc `c`) thì tự động `"bc"` và `"c"` cũng có mặt.

In [7]:
# Kiểm tra nhận xét: mọi chuỗi con hợp lệ kết thúc cùng chữ cái là đuôi của chuỗi dài nhất
def hop_le(t):
    return all((ord(t[i]) - ord(t[i - 1])) % 26 == 1 for i in range(1, len(t)))

s = "abcabc"
cac = {s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1) if hop_le(s[i:j])}
print(sorted(cac, key=lambda t: (t[-1], len(t))))     # ['a', 'b', 'ab', 'c', 'bc', 'abc']
print(len(cac))                                        # 6

['a', 'b', 'ab', 'c', 'bc', 'abc']
6


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi chuỗi con, đưa vào `set`

**Ý tưởng:** duyệt mọi cặp `(i, j)` → chuỗi con `s[i:j]`; kiểm tra nó có là chuỗi con của `base` không; nếu có thì bỏ vào `set`; đáp án là kích thước của `set`.

Để kiểm tra "có nằm trong `base` không" một cách **trực tiếp theo đề**, ta dựng một đoạn `base` đủ dài (bảng chữ cái lặp lại `len(s)//26 + 2` lần) rồi dùng toán tử `in`: `t in base`. Đoạn `base` hữu hạn này chứa mọi chuỗi con hợp lệ có độ dài `<= len(s)`.

In [8]:
def brute_set(s):
    base = string.ascii_lowercase * (len(s) // 26 + 2)
    co = set()
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            t = s[i:j]
            if t in base:                 # t có phải chuỗi con của base không?
                co.add(t)
    return len(co)

for s, expected in examples:
    print(s, brute_set(s), expected)
    assert brute_set(s) == expected
print(brute_set("abcabc"), brute_set("zabcdefghijklmnopqrstuvwxyzab"))

a 1 1
cac 2 2
zab 6 6
6 429


**Độ phức tạp:** có `O(n²)` chuỗi con; mỗi chuỗi mất `O(n)` để cắt lát, băm, và `t in base` (tìm chuỗi con) cũng `O(n)` trở lên. Tổng `O(n³)` thời gian; bộ nhớ `O(n²)` ký tự nếu mọi chuỗi con đều khác nhau... thậm chí tới `O(n³)` ký tự.

**Nút thắt:** hãy đo thời gian khi `n` tăng. Với `n = 10^5` là hoàn toàn bất khả thi.

In [9]:
random.seed(1)
for n in (50, 100, 200, 400):
    s = "".join(random.choice("abc") for _ in range(n))
    t0 = time.perf_counter()
    brute_set(s)
    print(f"n = {n:>3}: {1000 * (time.perf_counter() - t0):8.1f} ms")

n =  50:      0.2 ms
n = 100:      0.7 ms
n = 200:      3.5 ms
n = 400:     19.3 ms


Thời gian tăng gần **8 lần mỗi khi `n` gấp đôi** (đặc trưng của `O(n³)`). Ta cần một cách nhìn tốt hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Ba sự thật, mỗi sự thật dẫn ta một bước:

1. **Hợp lệ ⇔ mọi cặp kề nhau đều "kề nhau" trong vòng tròn** (mục 4). Vậy ta chỉ cần xét các **đoạn liên tiếp tối đa** của `s`, và mọi chuỗi con của chúng đều hợp lệ.
2. **Một chuỗi con hợp lệ được xác định hoàn toàn bởi cặp (chữ cái cuối, độ dài).** Vì sau khi biết chữ cuối là `c` và dài `L`, các ký tự trước đó bắt buộc là `c-1`, `c-2`, ... (theo vòng tròn). Ví dụ (cuối `c`, dài 3) chính là `"abc"`. Do đó hai chuỗi con hợp lệ khác nhau ⇔ cặp (cuối, dài) khác nhau.
3. **Nếu độ dài `L` kết thúc ở `c` có mặt thì mọi độ dài `1..L` kết thúc ở `c` cũng có mặt** (cắt bớt đầu). Vậy với mỗi chữ cái `c` chỉ cần biết `f[c]` = **độ dài dài nhất** của chuỗi con hợp lệ kết thúc ở `c`; số chuỗi con khác nhau kết thúc ở `c` đúng bằng `f[c]`.

**Đáp án = `f['a'] + f['b'] + ... + f['z']`.** Khi duyệt `s`, ta giữ `k` = độ dài đoạn liên tiếp đang kết thúc tại vị trí hiện tại: nếu ký tự hiện tại "kề sau" ký tự trước thì `k += 1`, ngược lại `k = 1`; rồi cập nhật `f[c] = max(f[c], k)`.

## 7. Các cách tiếp theo

### Cách 2 — Mở rộng từ mỗi vị trí bắt đầu, có cắt nhánh (vẫn dùng `set`)

**Ý tưởng:** từ mỗi chỉ số bắt đầu `i`, mở rộng sang phải **chừng nào còn kề nhau**; gặp cặp không hợp lệ thì **dừng hẳn** (mọi chuỗi con dài hơn từ `i` cũng sai). Không cần dùng `base` và toán tử `in` nữa. Vẫn đưa chuỗi con vào `set` để loại trùng.

**Giải thích từng đoạn:** vòng ngoài `i`; vòng trong `j` từ `i` tăng; trước khi thêm `s[i:j+1]` ta kiểm tra `s[j]` kề sau `s[j-1]` (nếu `j > i`); sai thì `break`.

In [10]:
def grow_set(s):
    co = set()
    for i in range(len(s)):
        for j in range(i, len(s)):
            if j > i and (ord(s[j]) - ord(s[j - 1])) % 26 != 1:
                break                      # đứt đoạn: mọi chuỗi dài hơn từ i đều sai
            co.add(s[i:j + 1])
    return len(co)

for s, expected in examples:
    assert grow_set(s) == expected
print(grow_set("abcabc"), grow_set("cac"))

6 2


In [11]:
class SolutionGrowSet:
    def findSubstringInWraproundString(self, s: str) -> int:
        seen = set()
        n = len(s)
        for i in range(n):
            for j in range(i, n):
                if j > i and (ord(s[j]) - ord(s[j - 1])) % 26 != 1:
                    break
                seen.add(s[i:j + 1])
        return len(seen)

print(SolutionGrowSet().findSubstringInWraproundString("zab"))

6


**Độ phức tạp (Cách 2):** nếu `s` là cả một đoạn liên tiếp dài `n`, vẫn có `~n²/2` chuỗi con, mỗi cái cắt lát `O(n)` → `O(n³)` thời gian trường hợp xấu nhất, `O(n³)` ký tự bộ nhớ. Tốt hơn Cách 1 khi chuỗi hay bị đứt, nhưng vẫn không qua nổi `n = 10^5`.

### Cách 3 — DP theo chữ cái kết thúc (cách chuẩn)

**Giải thích từng đoạn:**

- `f = [0] * 26`: `f[c]` là độ dài dài nhất của chuỗi con hợp lệ kết thúc ở chữ cái thứ `c`.
- `k = 0`: độ dài đoạn liên tiếp kết thúc ở vị trí đang xét.
- Với mỗi `i, c`: nếu `i > 0` và `(ord(c) - ord(s[i-1])) % 26 == 1` thì `k += 1`; ngược lại `k = 1` (bắt đầu đoạn mới, kể cả khi `i == 0`).
- `f[idx] = max(f[idx], k)`.
- Trả `sum(f)`.

In [12]:
def dp_end(s):
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
            k += 1
        else:
            k = 1
        idx = ord(c) - ord("a")
        f[idx] = max(f[idx], k)
    return sum(f)

for s, expected in examples:
    print(s, dp_end(s), expected)
    assert dp_end(s) == expected

a 1 1
cac 2 2
zab 6 6


In [13]:
class SolutionDPEnd:
    def findSubstringInWraproundString(self, s: str) -> int:
        f = [0] * 26
        k = 0
        for i, c in enumerate(s):
            if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
                k += 1
            else:
                k = 1
            idx = ord(c) - 97
            if k > f[idx]:
                f[idx] = k
        return sum(f)

print(SolutionDPEnd().findSubstringInWraproundString("cac"))

2


**Độ phức tạp (Cách 3):** một lượt duyệt `s` → `O(n)` thời gian; mảng 26 ô → `O(1)` (tức `O(|Σ|)`) bộ nhớ.

### Cách 4 — DP đối xứng: theo chữ cái **bắt đầu**, duyệt từ phải sang trái

**Ý tưởng khác:** thay vì cố định chữ cuối, ta cố định chữ **đầu**. Một chuỗi con hợp lệ cũng được xác định bởi cặp **(chữ cái đầu, độ dài)**. Với mỗi chữ cái đầu `c`, `g[c]` là độ dài dài nhất của chuỗi hợp lệ **bắt đầu** ở `c`; đáp án là `sum(g)`.

**Giải thích:** duyệt `i` từ `n-1` về `0`. `k` là độ dài đoạn liên tiếp **bắt đầu** ở `i`: nếu `s[i+1]` kề sau `s[i]` thì `k += 1`, ngược lại `k = 1`.

In [14]:
def dp_start(s):
    g = [0] * 26
    k = 0
    n = len(s)
    for i in range(n - 1, -1, -1):
        if i < n - 1 and (ord(s[i + 1]) - ord(s[i])) % 26 == 1:
            k += 1
        else:
            k = 1
        idx = ord(s[i]) - ord("a")
        g[idx] = max(g[idx], k)
    return sum(g)

for s, expected in examples:
    assert dp_start(s) == expected
print(dp_start("zab"), dp_start("abcabc"))

6 6


In [15]:
class SolutionDPStart:
    def findSubstringInWraproundString(self, s: str) -> int:
        g = [0] * 26
        k = 0
        n = len(s)
        for i in range(n - 1, -1, -1):
            if i < n - 1 and (ord(s[i + 1]) - ord(s[i])) % 26 == 1:
                k += 1
            else:
                k = 1
            idx = ord(s[i]) - 97
            if k > g[idx]:
                g[idx] = k
        return sum(g)

print(SolutionDPStart().findSubstringInWraproundString("zab"))

6


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(1)` bộ nhớ (26 ô). Cùng đáp án với Cách 3 do tính đối xứng: số cặp (đầu, dài) khác nhau bằng số cặp (cuối, dài) khác nhau, vì cả hai cùng đếm các chuỗi con hợp lệ khác nhau.

### Cách 5 — Chia thành các đoạn liên tiếp tối đa, chỉ xét 26 vị trí cuối mỗi đoạn

**Ý tưởng khác:** cắt `s` thành các **đoạn liên tiếp tối đa** (hai ký tự kề nhau thuộc cùng đoạn nếu kề nhau trong vòng tròn). Trong một đoạn dài `L` bắt đầu tại `l`, vị trí `p` có `k = p - l + 1`. Mỗi chữ cái chỉ xuất hiện lại sau mỗi 26 bước, và lần xuất hiện **sau cùng** của nó trong đoạn có `k` lớn nhất. Do đó chỉ cần xét tối đa **26 vị trí cuối** của mỗi đoạn.

**Giải thích:** hai con trỏ: `l` là đầu đoạn; `r` đi tìm hết đoạn. Với mỗi đoạn `[l, r)`, duyệt `p` từ `max(l, r - 26)` đến `r - 1` và cập nhật `f`.

In [16]:
def by_runs(s):
    f = [0] * 26
    n = len(s)
    l = 0
    while l < n:
        r = l + 1
        while r < n and (ord(s[r]) - ord(s[r - 1])) % 26 == 1:
            r += 1                              # đoạn liên tiếp tối đa là s[l:r]
        for p in range(max(l, r - 26), r):      # chỉ 26 vị trí cuối có thể là kỷ lục
            idx = ord(s[p]) - ord("a")
            f[idx] = max(f[idx], p - l + 1)
        l = r
    return sum(f)

for s, expected in examples:
    assert by_runs(s) == expected
print(by_runs("zab"), by_runs("abcdefghijklmnopqrstuvwxyzabc"))

6 429


In [17]:
class SolutionRuns:
    def findSubstringInWraproundString(self, s: str) -> int:
        f = [0] * 26
        n = len(s)
        l = 0
        while l < n:
            r = l + 1
            while r < n and (ord(s[r]) - ord(s[r - 1])) % 26 == 1:
                r += 1
            for p in range(max(l, r - 26), r):
                idx = ord(s[p]) - 97
                f[idx] = max(f[idx], p - l + 1)
            l = r
        return sum(f)

print(SolutionRuns().findSubstringInWraproundString("zab"))

6


**Độ phức tạp (Cách 5):** mỗi ký tự được quét một lần để tìm đoạn (`O(n)`), rồi mỗi đoạn xử lý tối đa 26 vị trí (`<= n` tổng cộng) → `O(n)` thời gian, `O(1)` bộ nhớ. Ưu điểm: tách bạch "tìm đoạn" và "cập nhật kỷ lục", dễ nhìn bản chất.

## 8. TRACE từng bước

Trace **Cách 3** với `s = "zabcab"`. Quy ước: `k` là độ dài đoạn liên tiếp kết thúc tại `i`; `f[c]` là kỷ lục theo chữ cái cuối.

| i | `c` | kề sau ký tự trước? | `k` | `f` sau bước (chỉ các ô khác 0) | tổng `sum(f)` |
|---|---|---|---|---|---|
| 0 | z | (không có trước) | 1 | z=1 | 1 |
| 1 | a | z→a: (0-25)%26=1 ✔ | 2 | z=1, a=2 | 3 |
| 2 | b | a→b ✔ | 3 | z=1, a=2, b=3 | 6 |
| 3 | c | b→c ✔ | 4 | z=1, a=2, b=3, c=4 | 10 |
| 4 | a | c→a: (0-2)%26=24 ✘ | 1 | `f[a]=max(2,1)=2` (không đổi) | 10 |
| 5 | b | a→b ✔ | 2 | `f[b]=max(3,2)=3` (không đổi) | 10 |

Đáp án `10`: các chuỗi con là `z, a, b, c, za, ab, bc, zab, abc, zabc`. Đoạn `"ab"` ở cuối **trùng** với `"ab"` đã đếm trong `"zabc"`, nên không cộng thêm.

In [18]:
def trace_dp(s):
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
            k += 1
            ly_do = f"{s[i-1]}->{c} kề nhau, k tăng"
        else:
            k = 1
            ly_do = "đứt đoạn hoặc bắt đầu, k = 1"
        idx = ord(c) - ord("a")
        cu = f[idx]
        f[idx] = max(f[idx], k)
        khac0 = {chr(97 + t): v for t, v in enumerate(f) if v}
        print(f"i={i} c={c!r} | {ly_do:<28} | k={k} | f[{c}]: {cu}->{f[idx]} | f={khac0} | sum={sum(f)}")
    return sum(f)

print("Đáp án:", trace_dp("zabcab"))

i=0 c='z' | đứt đoạn hoặc bắt đầu, k = 1 | k=1 | f[z]: 0->1 | f={'z': 1} | sum=1
i=1 c='a' | z->a kề nhau, k tăng         | k=2 | f[a]: 0->2 | f={'a': 2, 'z': 1} | sum=3
i=2 c='b' | a->b kề nhau, k tăng         | k=3 | f[b]: 0->3 | f={'a': 2, 'b': 3, 'z': 1} | sum=6
i=3 c='c' | b->c kề nhau, k tăng         | k=4 | f[c]: 0->4 | f={'a': 2, 'b': 3, 'c': 4, 'z': 1} | sum=10
i=4 c='a' | đứt đoạn hoặc bắt đầu, k = 1 | k=1 | f[a]: 2->2 | f={'a': 2, 'b': 3, 'c': 4, 'z': 1} | sum=10
i=5 c='b' | a->b kề nhau, k tăng         | k=2 | f[b]: 3->3 | f={'a': 2, 'b': 3, 'c': 4, 'z': 1} | sum=10
Đáp án: 10


In [19]:
# Trace thêm ví dụ có lặp: "abcabc"
print("Đáp án:", trace_dp("abcabc"))

i=0 c='a' | đứt đoạn hoặc bắt đầu, k = 1 | k=1 | f[a]: 0->1 | f={'a': 1} | sum=1
i=1 c='b' | a->b kề nhau, k tăng         | k=2 | f[b]: 0->2 | f={'a': 1, 'b': 2} | sum=3
i=2 c='c' | b->c kề nhau, k tăng         | k=3 | f[c]: 0->3 | f={'a': 1, 'b': 2, 'c': 3} | sum=6
i=3 c='a' | đứt đoạn hoặc bắt đầu, k = 1 | k=1 | f[a]: 1->1 | f={'a': 1, 'b': 2, 'c': 3} | sum=6
i=4 c='b' | a->b kề nhau, k tăng         | k=2 | f[b]: 2->2 | f={'a': 1, 'b': 2, 'c': 3} | sum=6
i=5 c='c' | b->c kề nhau, k tăng         | k=3 | f[c]: 3->3 | f={'a': 1, 'b': 2, 'c': 3} | sum=6
Đáp án: 6


## 9. Code nộp LeetCode

Dùng Cách 3 (DP theo chữ cái kết thúc): ngắn, nhanh, `O(n)` thời gian và `O(1)` bộ nhớ.

In [20]:
class Solution:
    def findSubstringInWraproundString(self, s: str) -> int:
        f = [0] * 26          # f[c] = độ dài dài nhất của chuỗi con hợp lệ kết thúc bằng chữ cái c
        k = 0                 # độ dài đoạn liên tiếp kết thúc tại vị trí hiện tại
        for i, c in enumerate(s):
            if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
                k += 1
            else:
                k = 1
            idx = ord(c) - 97
            if k > f[idx]:
                f[idx] = k
        return sum(f)


for s, expected in examples:
    print(s, Solution().findSubstringInWraproundString(s), expected)
    assert Solution().findSubstringInWraproundString(s) == expected

a 1 1
cac 2 2
zab 6 6


## 10. Vì sao thuật toán đúng

**Mệnh đề 1.** Một chuỗi `t` là chuỗi con của `base` khi và chỉ khi với mọi `i >= 1`: `(ord(t[i]) - ord(t[i-1])) % 26 == 1`. (`base` là đường đi quanh vòng tròn, mỗi bước tiến đúng 1 ô.)

**Mệnh đề 2.** Các chuỗi con hợp lệ được xác định bởi (chữ cuối `c`, độ dài `L`): ký tự thứ `L - 1 - j` tính từ cuối là chữ cái cách `c` lùi `j` bước trên vòng tròn. Vậy hai chuỗi con hợp lệ bằng nhau ⇔ cùng (cuối, dài).

**Mệnh đề 3.** Nếu có chuỗi con hợp lệ cuối `c` dài `L` thì có chuỗi cuối `c` dài `L'` với mọi `1 <= L' <= L` (bỏ bớt các ký tự đầu vẫn hợp lệ). Số cặp (c, L') có mặt cho `c` cố định là đúng `max L` = `f[c]`.

**Chứng minh `k` đúng.** `k` ở vị trí `i` là độ dài đoạn liên tiếp dài nhất kết thúc tại `i` (quy nạp: nếu `s[i]` kề sau `s[i-1]` thì kéo dài thêm 1, ngược lại chỉ còn chính nó). Các chuỗi con hợp lệ kết thúc tại `i` có độ dài đúng từ `1` đến `k`. Lấy `max` qua mọi vị trí có cùng chữ cuối `c` ra `f[c]`. Tổng `sum(f)` là số cặp (c, L) khác nhau = số chuỗi con hợp lệ khác nhau. ∎

In [21]:
# Kiểm chứng Mệnh đề 2 bằng thực nghiệm: các chuỗi con hợp lệ khác nhau <-> các cặp (cuối, dài) khác nhau
random.seed(7)
for _ in range(200):
    s = "".join(random.choice("abcz") for _ in range(12))
    hop = {s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1) if hop_le(s[i:j])}
    cap = {(t[-1], len(t)) for t in hop}
    assert len(hop) == len(cap) == dp_end(s)
print("Mệnh đề 2 và đáp án khớp trên 200 chuỗi ngẫu nhiên")

Mệnh đề 2 và đáp án khớp trên 200 chuỗi ngẫu nhiên


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Set mọi chuỗi con + `in base` | `O(n³)` | `O(n³)` ký tự |
| 2. Mở rộng có cắt nhánh + set | `O(n³)` xấu nhất | `O(n³)` ký tự |
| 3. DP theo chữ cuối | `O(n)` | `O(1)` (26 ô) |
| 4. DP theo chữ đầu | `O(n)` | `O(1)` |
| 5. Chia đoạn, 26 vị trí cuối | `O(n)` | `O(1)` |

Với `n = 10^5`: Cách 3/4/5 chạy trong chừng chục mili-giây; Cách 1/2 cần hàng tỷ thao tác ký tự.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [22]:
sol = Solution()
f467 = sol.findSubstringInWraproundString

for s, expected in examples:
    assert f467(s) == expected

assert f467("a") == 1                              # một chữ cái
assert f467("aaaa") == 1                           # toàn chữ trùng: chỉ "a"
assert f467("abc") == 6                            # 3 + 2 + 1
assert f467("cba") == 3                            # ngược thứ tự: chỉ chuỗi 1 ký tự
assert f467("za") == 3                             # z, a, za
assert f467("az") == 2                             # a→z không kề: chỉ "a" và "z"
assert f467("abcabc") == 6                         # đoạn "abc" lặp, không đếm đôi
assert f467("abcdefghijklmnopqrstuvwxyz") == 351   # 26*27/2
print("Ví dụ và edge case OK")

Ví dụ và edge case OK


### 12.2. Phản ví dụ cho các lời giải sai thường gặp

`"abcabc"` đúng là `6`; nếu cộng dồn mọi `k` ta ra `12`; `"zab"` đúng là `6`; nếu quên `z→a` ta ra `3`.

In [23]:
def sai_cong_don(s):                    # SAI: cộng k mỗi bước, không lấy max theo chữ cái
    tong = 0
    k = 0
    for i, c in enumerate(s):
        k = k + 1 if i and (ord(c) - ord(s[i - 1])) % 26 == 1 else 1
        tong += k
    return tong

def sai_quen_wrap(s):                   # SAI: chỉ chấp nhận hiệu == 1 (quên z -> a)
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        k = k + 1 if i and ord(c) - ord(s[i - 1]) == 1 else 1
        f[ord(c) - 97] = max(f[ord(c) - 97], k)
    return sum(f)

print(sai_cong_don("abcabc"), "(đúng là 6)")
print(sai_quen_wrap("zab"), "(đúng là 6)")
assert sai_cong_don("abcabc") != 6 and sai_quen_wrap("zab") != 6

12 (đúng là 6)
4 (đúng là 6)


### 12.3. Đối chiếu các cách bằng ngẫu nhiên có seed

In [24]:
random.seed(2024)
alphabets = ["a", "ab", "abc", "az", "xyza", string.ascii_lowercase]
dem = 0
for _ in range(1500):
    alpha = random.choice(alphabets)
    n = random.randint(1, 40)
    # trộn: lúc thì ký tự ngẫu nhiên, lúc thì nối tiếp theo bảng chữ cái (để có đoạn liên tiếp dài)
    s = []
    for _ in range(n):
        if s and random.random() < 0.6:
            s.append(chr((ord(s[-1]) - 97 + 1) % 26 + 97))
        else:
            s.append(random.choice(alpha))
    s = "".join(s)
    ket_qua = {
        brute_set(s), grow_set(s), dp_end(s), dp_start(s), by_runs(s),
        SolutionGrowSet().findSubstringInWraproundString(s),
        SolutionDPEnd().findSubstringInWraproundString(s),
        SolutionDPStart().findSubstringInWraproundString(s),
        SolutionRuns().findSubstringInWraproundString(s),
        f467(s),
    }
    assert len(ket_qua) == 1, (s, ket_qua)
    dem += 1
print("Đã đối chiếu", dem, "chuỗi: tất cả các cách đều khớp với vét cạn")

Đã đối chiếu 1500 chuỗi: tất cả các cách đều khớp với vét cạn


### 12.4. Test chuỗi dài (ràng buộc `n = 10^5`)

Một chuỗi dài 100 000 ký tự **toàn đoạn liên tiếp**: mỗi chữ cái `c` có `f[c]` lớn. Ta tự tính đáp án bằng công thức đóng để đối chiếu: với `s` là bảng chữ cái lặp, bắt đầu từ `a`, dài `n`, thì `f[c]` là độ dài lớn nhất kết thúc ở `c`: đó là vị trí cuối cùng (tính từ 1) của chữ `c` trong `s`.

In [25]:
n = 100_000
s_dai = "".join(chr(97 + i % 26) for i in range(n))
t0 = time.perf_counter()
kq = f467(s_dai)
t1 = time.perf_counter()
# công thức: với chữ cái c (vị trí 0..25), số lần xuất hiện cuối cùng ở vị trí p (0-indexed) lớn nhất < n  =>  k = p + 1
cong_thuc = 0
for c in range(26):
    p = max(i for i in range(c, n, 26))
    cong_thuc += p + 1
print(kq, cong_thuc, f"{1000 * (t1 - t0):.1f} ms")
assert kq == cong_thuc
assert f467("a" * n) == 1
print("ab lặp 50000 lần:", f467("ab" * 50_000))      # a->b ✔, b->a ✘ => chỉ a, b, ab = 3
assert f467("ab" * 50_000) == 3

2599675 2599675 13.8 ms
ab lặp 50000 lần: 3


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp vòng `z → a`

Chỉ so `ord(c) - ord(prev) == 1` thì `"za"` bị coi là đứt đoạn.

In [26]:
def loi1_sai(s):
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        k = k + 1 if i and ord(c) - ord(s[i - 1]) == 1 else 1     # SAI: thiếu wrap
        f[ord(c) - 97] = max(f[ord(c) - 97], k)
    return sum(f)

print("sai :", loi1_sai("za"), " (mong đợi 3: z, a, za)")
print("đúng:", f467("za"))
assert loi1_sai("za") == 2 and f467("za") == 3

sai : 2  (mong đợi 3: z, a, za)
đúng: 3


**Vì sao sai:** `ord('a') - ord('z') = -25`, khác `1`. **Cách đúng:** dùng `% 26`, hoặc so sánh thêm trường hợp `prev == 'z' and c == 'a'`.

### Lỗi 2 — Cộng dồn `k` thay vì lấy `max` theo từng chữ cái

Đếm `k` mỗi bước thì các đoạn lặp lại bị đếm hai lần.

In [27]:
print("sai :", sai_cong_don("abcabc"), "(đếm trùng)")
print("đúng:", f467("abcabc"))
assert sai_cong_don("abcabc") == 12 and f467("abcabc") == 6

sai : 12 (đếm trùng)
đúng: 6


**Vì sao sai:** `"abcabc"` có hai đoạn `"abc"` giống nhau; mỗi đoạn góp `1+2+3 = 6`, tổng `12`. Nhưng chuỗi con **khác nhau** chỉ có 6. **Cách đúng:** chỉ giữ **kỷ lục** `f[c]` (mục 6, sự thật 3).

### Lỗi 3 — Quên reset `k = 1` khi bị đứt (hoặc reset về `0`)

Reset về `0` làm mất chính ký tự hiện tại, đáp án nhỏ hơn đúng.

In [28]:
def loi3_sai(s):
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
            k += 1
        else:
            k = 0                          # SAI: phải là 1 (chính ký tự c cũng là chuỗi con)
        f[ord(c) - 97] = max(f[ord(c) - 97], k)
    return sum(f)

print("sai :", loi3_sai("cac"), " (mong đợi 2)")
print("đúng:", f467("cac"))
assert loi3_sai("cac") == 0 and f467("cac") == 2

sai : 0  (mong đợi 2)
đúng: 2


**Vì sao sai:** sau khi đứt đoạn, ký tự `c` hiện tại vẫn bắt đầu một đoạn mới có độ dài `1`. **Cách đúng:** `k = 1`.

### Lỗi 4 — Dùng `set` chuỗi con cho `n = 10^5`

Cách 1/2 đúng về kết quả nhưng tốn `O(n³)`: với `n = 2000` đã chậm, `n = 10^5` không thể.

In [29]:
random.seed(3)
s = "".join(chr(97 + i % 26) for i in range(600))
t0 = time.perf_counter(); a = grow_set(s); t1 = time.perf_counter(); b = f467(s); t2 = time.perf_counter()
print(f"n = 600: grow_set {1000 * (t1 - t0):.0f} ms, Solution {1000 * (t2 - t1):.3f} ms, cùng đáp án: {a == b}")
assert a == b

n = 600: grow_set 49 ms, Solution 0.122 ms, cùng đáp án: True


**Cách đúng:** dùng mảng 26 ô, không lưu chuỗi con nào.

### Lỗi 5 (nhỏ) — Đếm cả chuỗi rỗng hoặc nhầm "số lần xuất hiện" với "số chuỗi khác nhau"

Đề hỏi số chuỗi con **khác nhau, không rỗng**. Nếu ai đó tính `n(n+1)/2` cho đoạn toàn hợp lệ mà quên rằng vòng tròn có thể làm hai đoạn con **trùng nhau** (ví dụ khi `n > 26`, các chuỗi con dài cùng (cuối, dài) khác vị trí thì vẫn là một), sẽ ra sai.

In [30]:
s = string.ascii_lowercase * 2            # 52 ký tự, toàn bộ liên tiếp
print("n(n+1)/2 =", 52 * 53 // 2, "nhưng đáp án thật:", f467(s))
# chữ cái thứ c xuất hiện lần cuối ở vị trí 26 + c (đếm từ 0) nên có kỷ lục k = 27 + c
assert f467(s) == sum(27 + c for c in range(26)) == 1027

n(n+1)/2 = 1378 nhưng đáp án thật: 1027


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm chuỗi con **khác nhau** thỏa tính chất cục bộ giữa hai ký tự kề nhau" + `n` lớn → đừng liệt kê chuỗi con; hãy tìm **khóa định danh** gọn (ở đây: cặp chữ cuối, độ dài) rồi đếm khóa.
- **Mẫu "độ dài dài nhất kết thúc tại ..."** (`longest ending here`) là tư duy DP quen thuộc: `k` ở đây giống `dp[i]`, còn `f[c]` là "gộp theo nhóm" để loại trùng.
- **Kiểm tra "kề nhau trên vòng tròn"** = `(a - b) % m == 1`; với `m = 26` là bảng chữ cái, tương tự đồng hồ 12 giờ, ngày trong tuần...
- **Nếu "tính chất" chỉ phụ thuộc cặp kề nhau**, hãy chia chuỗi thành các **đoạn liên tiếp tối đa**; mọi chuỗi con của đoạn tối đa đều thỏa.
- **Nếu bị hỏi "trùng" thì nghĩ tới nhóm:** nhóm các chuỗi con theo đặc trưng và chỉ giữ đại diện lớn nhất (vì mọi cái nhỏ hơn là hậu tố của nó).

## 15. Biến thể

**Biến thể A — bước nhảy khác 1.** Nếu "liên tiếp" nghĩa là tăng đúng `d` (mod 26) thay vì `1`, chỉ cần thay `== 1` bằng `== d`. Tổng quát hóa với bảng chữ cái có `m` ký tự: `% m`.

**Biến thể B — cho phép cả tăng và giảm.** Chuỗi con của `base` hai chiều (tăng `+1` hoặc giảm `-1` liên tiếp nhất quán): xử lý hai hướng riêng và cộng, nhớ trừ phần chuỗi 1 ký tự bị đếm hai lần.

**Biến thể C — đếm tất cả (kể cả trùng).** Đáp án đơn giản hơn: cộng `k` mỗi bước (chính là code "sai" ở Lỗi 2, nay trở thành đúng cho đề khác!).

In [31]:
def dem_khac_nhau_buoc(s, d):
    """Biến thể A: đếm chuỗi con khác nhau mà mỗi ký tự = ký tự trước + d (mod 26)."""
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        k = k + 1 if i and (ord(c) - ord(s[i - 1])) % 26 == d % 26 else 1
        f[ord(c) - 97] = max(f[ord(c) - 97], k)
    return sum(f)

def dem_ca_trung(s):
    """Biến thể C: đếm mọi chuỗi con hợp lệ, tính cả vị trí khác nhau."""
    return sai_cong_don(s)

print(dem_khac_nhau_buoc("zab", 1), dem_khac_nhau_buoc("acegik", 2), dem_khac_nhau_buoc("acegik", 1))
print(dem_ca_trung("abcabc"))
assert dem_khac_nhau_buoc("zab", 1) == 6
assert dem_khac_nhau_buoc("acegik", 2) == 21
assert dem_ca_trung("abcabc") == 12

6 21 6
12


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Set + `in base` | Liệt kê mọi chuỗi con, kiểm tra theo định nghĩa | `O(n³)` | `O(n³)` | Làm oracle để đối chiếu, `n` rất nhỏ |
| 2. Mở rộng + cắt nhánh | Từ mỗi điểm đầu, kéo dài đến khi đứt | `O(n³)` xấu nhất | `O(n³)` | Cải tiến dễ nghĩ nhưng chưa đủ |
| 3. DP chữ cuối | `f[c] = max k`, đáp án `sum(f)` | `O(n)` | `O(1)` | **Cách nộp bài** |
| 4. DP chữ đầu | Đối xứng, duyệt từ phải sang trái | `O(n)` | `O(1)` | Kiểm tra chéo, minh họa đối xứng |
| 5. Đoạn tối đa + 26 cuối | Tách đoạn, chỉ xét 26 vị trí cuối | `O(n)` | `O(1)` | Khi muốn thấy rõ cấu trúc đoạn |

In [32]:
s = "".join(chr(97 + (i // 7 * 5 + i) % 26) for i in range(100_000))
cac = [("Cách 3 (chữ cuối)", dp_end), ("Cách 4 (chữ đầu)", dp_start), ("Cách 5 (đoạn)", by_runs)]
kq = []
for ten, fn in cac:
    t0 = time.perf_counter()
    v = fn(s)
    kq.append(v)
    print(f"{ten:<20} -> {v}  {1000 * (time.perf_counter() - t0):7.1f} ms")
assert len(set(kq)) == 1

Cách 3 (chữ cuối)    -> 169     12.5 ms
Cách 4 (chữ đầu)     -> 169     14.2 ms
Cách 5 (đoạn)        -> 169     16.0 ms


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `findSubstringInWraproundString("abcd")`; (b) `("zyx")`; (c) `("abab")`; (d) `(ord("a") - ord("z")) % 26`; (e) `("xyzabc")`.

**Bài 2 (tính tay).** Với `s = "yzabxyz"`, lập bảng `i, c, k, f` như mục 8 và tìm đáp án.

**Bài 3 (điền TODO).** Hoàn thành `dem_todo(s)` theo Cách 3.

In [33]:
def dem_todo(s):
    # TODO 1: tạo mảng f gồm 26 số 0 và biến k = 0
    # TODO 2: for i, c in enumerate(s): nếu i > 0 và (ord(c) - ord(s[i-1])) % 26 == 1 thì k += 1 ngược lại k = 1
    # TODO 3: cập nhật f[ord(c) - 97] = max(f[ord(c) - 97], k)
    # TODO 4: return sum(f)
    return None

# KHÔNG gọi dem_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `dem_phai_sang_trai(s)` theo Cách 4 (chữ cái **bắt đầu**, duyệt từ phải) **mà không nhìn lại** mục 7. Kiểm tra `"zab" → 6`, `"cac" → 2`.

**Bài 5 (biến thể / mini challenge).** Bảng chữ cái giờ chỉ có `m` chữ cái đầu tiên (ví dụ `m = 5`: `abcde` lặp vô hạn, sau `e` là `a`). Chuỗi `s` chỉ gồm các chữ cái trong `m` chữ cái đó. Viết `dem_m(s, m)`. Kiểm tra `dem_m("eabcd", 5)` và `dem_m("ace", 5)`. Gợi ý: thay `26` bằng `m`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: một chuỗi toàn đoạn liên tiếp dài `L` cho `L(L+1)/2` (khi `L <= 26`); `"abab"`: `a→b` ✔, `b→a` ✘. Bài 3: bắt đầu với `f = [0] * 26`.
- **Mức 2 (vừa):** Bài 2: `y→z` ✔, `z→a` ✔, `a→b` ✔, `b→x` ✘, `x→y` ✔, `y→z` ✔. Có hai đoạn liên tiếp tối đa `"yzab"` và `"xyz"`. Bài 4: `k` là độ dài đoạn **bắt đầu** tại `i`.
- **Mức 3 (sâu):** Bài 5: hiệu `(ord(c) - ord(p)) % m == 1` với `ord` tính từ `'a'`. Nhớ rằng chữ cái nằm trong `m` ô đầu nên mảng `f` có `m` ô, còn đáp án cho đoạn dài sẽ tăng theo `m`.

## 19. Đáp án tham khảo cho bài tập

In [34]:
# Bài 1
assert f467("abcd") == 10
assert f467("zyx") == 3
assert f467("abab") == 3
assert (ord("a") - ord("z")) % 26 == 1
assert f467("xyzabc") == 21
print("Bài 1 OK")

# Bài 2: "yzabxyz"
print("Bài 2:")
print(trace_dp("yzabxyz"))
# đoạn "yzab" cho y=1,z=2,a=3,b=4 -> 10 ; đoạn "xyz": x=1, y=max(1,2)=2, z=max(2,3)=3 -> x:1, y:2, z:3 ; tổng = 1 + 2 + 3 + 3 + 4 = 13
assert f467("yzabxyz") == 13

# Bài 3
def dem_todo_ans(s):
    f = [0] * 26
    k = 0
    for i, c in enumerate(s):
        if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
            k += 1
        else:
            k = 1
        f[ord(c) - 97] = max(f[ord(c) - 97], k)
    return sum(f)

random.seed(5)
for _ in range(100):
    s = "".join(random.choice("abcxyz") for _ in range(random.randint(1, 30)))
    assert dem_todo_ans(s) == f467(s) == brute_set(s)
print("Bài 3 OK")

# Bài 4
def dem_phai_sang_trai(s):
    g = [0] * 26
    k = 0
    for i in range(len(s) - 1, -1, -1):
        if i + 1 < len(s) and (ord(s[i + 1]) - ord(s[i])) % 26 == 1:
            k += 1
        else:
            k = 1
        g[ord(s[i]) - 97] = max(g[ord(s[i]) - 97], k)
    return sum(g)

assert dem_phai_sang_trai("zab") == 6 and dem_phai_sang_trai("cac") == 2
for _ in range(100):
    s = "".join(random.choice("abcxyz") for _ in range(random.randint(1, 30)))
    assert dem_phai_sang_trai(s) == f467(s)
print("Bài 4 OK")

# Bài 5
def dem_m(s, m):
    f = [0] * m
    k = 0
    for i, c in enumerate(s):
        if i > 0 and (ord(c) - ord(s[i - 1])) % m == 1:
            k += 1
        else:
            k = 1
        f[ord(c) - 97] = max(f[ord(c) - 97], k)
    return sum(f)

print("dem_m('eabcd', 5) =", dem_m("eabcd", 5), "; dem_m('ace', 5) =", dem_m("ace", 5))
assert dem_m("eabcd", 5) == 15           # e→a ✔ (wrap theo m=5): cả 5 ký tự liên tiếp: 5+4+3+2+1
assert dem_m("ace", 5) == 3              # a→c ✘, c→e ✘, chỉ 3 chữ riêng lẻ
assert dem_m("abcdeabcde", 5) == 6 + 7 + 8 + 9 + 10      # cả 10 ký tự liên tiếp: kỷ lục mỗi chữ là 6..10
print("Bài 5 OK")

Bài 1 OK
Bài 2:
i=0 c='y' | đứt đoạn hoặc bắt đầu, k = 1 | k=1 | f[y]: 0->1 | f={'y': 1} | sum=1
i=1 c='z' | y->z kề nhau, k tăng         | k=2 | f[z]: 0->2 | f={'y': 1, 'z': 2} | sum=3
i=2 c='a' | z->a kề nhau, k tăng         | k=3 | f[a]: 0->3 | f={'a': 3, 'y': 1, 'z': 2} | sum=6
i=3 c='b' | a->b kề nhau, k tăng         | k=4 | f[b]: 0->4 | f={'a': 3, 'b': 4, 'y': 1, 'z': 2} | sum=10
i=4 c='x' | đứt đoạn hoặc bắt đầu, k = 1 | k=1 | f[x]: 0->1 | f={'a': 3, 'b': 4, 'x': 1, 'y': 1, 'z': 2} | sum=11
i=5 c='y' | x->y kề nhau, k tăng         | k=2 | f[y]: 1->2 | f={'a': 3, 'b': 4, 'x': 1, 'y': 2, 'z': 2} | sum=12
i=6 c='z' | y->z kề nhau, k tăng         | k=3 | f[z]: 2->3 | f={'a': 3, 'b': 4, 'x': 1, 'y': 2, 'z': 3} | sum=13
13
Bài 3 OK
Bài 4 OK
dem_m('eabcd', 5) = 15 ; dem_m('ace', 5) = 3
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (đếm khóa duy nhất theo "độ dài dài nhất kết thúc tại nhóm"):**

```python
f = [0] * 26
k = 0
for i, c in enumerate(s):
    if i > 0 and (ord(c) - ord(s[i - 1])) % 26 == 1:
        k += 1
    else:
        k = 1
    idx = ord(c) - 97
    f[idx] = max(f[idx], k)
return sum(f)
```

**Checklist:**

- [ ] Kiểm tra "kề nhau" bằng `% 26 == 1` (có xử lý `z → a`).
- [ ] Reset `k = 1` (không phải `0`) khi đứt đoạn hoặc ở `i = 0`.
- [ ] Cập nhật bằng `max` theo **chữ cái kết thúc**, **không** cộng dồn `k`.
- [ ] Trả `sum(f)`, không phải `max` hay `len`.
- [ ] Thử tay `"a" → 1`, `"cac" → 2`, `"zab" → 6`, `"abcabc" → 6`.
- [ ] Đúng tên method `findSubstringInWraproundString(self, s: str) -> int` (có lỗi chính tả `Wrapround` của LeetCode).
- [ ] Độ phức tạp `O(n)` thời gian, `O(1)` bộ nhớ.